## Setup

In [2]:
import pandas as pd
import numpy as np
import os
from langchain_community.document_loaders import PyPDFLoader, UnstructuredPDFLoader, PyPDFium2Loader
from langchain.document_loaders import PyPDFDirectoryLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path
import random
import ollama
import networkx as nx


## Input data directory
data_dir = "PFAS_test"
inputdirectory = Path(f"./data_input/{data_dir}")
## This is where the output csv files will be written
out_dir = data_dir
outputdirectory = Path(f"./data_output/{out_dir}")

## Load Documents

In [3]:
## Dir PDF Loader
#loader = PyPDFDirectoryLoader(inputdirectory)
## File Loader
#loader = PyPDFLoader("./data/MedicalDocuments/orf-path_health-n1.pdf")
loader = DirectoryLoader(inputdirectory, show_progress=True)
documents = loader.load()

100%|██████████| 1/1 [00:01<00:00,  1.40s/it]


## Extract Concepts

In [4]:
## This function uses the helpers/prompt function to extract concepts from text
from helpers.df_helpers import docs2Graph

If regenerate is set to True then the dataframes are regenerated and Both the dataframes are written in the csv format so we dont have to calculate them again. 

        dfne = dataframe of edges

        df = dataframe of chunks


Else the dataframes are read from the output directory

In [5]:
# To regenerate the graph with LLM, set this to True
regenerate = False

if regenerate:
    # Generate concepts list from full text documents (no chunking)
    G_graph = docs2Graph(documents, model='zephyr:latest')
    # Convert to DataFrame from the graph
    dfg1 = nx.to_pandas_edgelist(G_graph)
    #save the graph
    nx.write_graphml(G_graph, "output_graph.graphml")

else: 
    G_graph = nx.read_graphml("output_graph.graphml")
    dfg1 = nx.to_pandas_edgelist(G_graph)

In [6]:
print(type(G_graph))


<class 'networkx.classes.digraph.DiGraph'>


### Calculate communities for coloring the nodes

In [7]:
communities_generator = nx.community.girvan_newman(G_graph)
top_level_communities = next(communities_generator)
next_level_communities = next(communities_generator)
communities = sorted(map(sorted, next_level_communities))
print("Number of Communities = ", len(communities))
print(communities)

Number of Communities =  11
[['PDMS'], ['carbon black'], ['ceramic nanoparticles'], ['chitosan'], ['genipin'], ['hydrogen peroxide'], ['iron(II) sulfate', 'pH buffering agents'], ['lactic acid', 'zinc oxide'], ['lithium salt'], ['plasticizers'], ['polyethylene oxide']]


### Create a dataframe for community colors

In [8]:
import seaborn as sns
palette = "hls"

## Now add these colors to communities and make another dataframe
def colors2Community(communities) -> pd.DataFrame:
    ## Define a color palette
    p = sns.color_palette(palette, len(communities)).as_hex()
    random.shuffle(p)
    rows = []
    group = 0
    for community in communities:
        color = p.pop()
        group += 1
        for node in community:
            rows += [{"node": node, "color": color, "group": group}]
    df_colors = pd.DataFrame(rows)
    return df_colors


colors = colors2Community(communities)
colors

,node,color,group
0,PDMS,#db5797,1
1,carbon black,#db5f57,2
2,ceramic nanoparticles,#7fdb57,3
3,chitosan,#d757db,4
4,genipin,#c7db57,5
5,hydrogen peroxide,#57dbbf,6
6,iron(II) sulfate,#5767db,7
7,pH buffering agents,#5767db,7
8,lactic acid,#dba757,8
9,zinc oxide,#dba757,8


### Add colors to the graph

In [9]:
for index, row in colors.iterrows():
    G_graph.nodes[row['node']]['group'] = row['group']
    G_graph.nodes[row['node']]['color'] = row['color']
    G_graph.nodes[row['node']]['size'] = G_graph.degree[row['node']]

In [1]:
from pyvis.network import Network

graph_output_directory = "./docs/index.html"

net = Network(
    notebook=False,
    # bgcolor="#1a1a1a",
    cdn_resources="remote",
    height="900px",
    width="100%",
    select_menu=True,
    # font_color="#cccccc",
    filter_menu=False,
)

net.from_nx(G_graph)
# net.repulsion(node_distance=150, spring_length=400)
net.force_atlas_2based(central_gravity=0.015, gravity=-31)
# net.barnes_hut(gravity=-18100, central_gravity=5.05, spring_length=380)
net.show_buttons(filter_=["physics"])

net.show(graph_output_directory, notebook=False)

NameError: name 'G_graph' is not defined